# IRL Alignment Auditor - Example Usage

This notebook demonstrates how to use the IRL Alignment Auditor pipeline for auditing and refining LLM objectives using Bayesian Inverse Reinforcement Learning.

**Paper**: [The Alignment Auditor: A Bayesian Framework for Verifying and Refining LLM Objectives](https://arxiv.org/abs/2510.06096)

**GitHub**: https://github.com/Matthieu6/IRL-Alignment-Auditor

## Setup

First, let's clone the repository and install dependencies:


In [ ]:
# Clone the repository
!git clone https://github.com/Matthieu6/IRL-Alignment-Auditor.git
%cd IRL-Alignment-Auditor


In [ ]:
# Install dependencies
%pip install -r requirements.txt


In [ ]:
# Verify installation
!python -c "import irl_pipeline; print('Installation successful!')"


## Hugging Face Token Setup

**Important**: You need a Hugging Face token to download models and save results to the Hub.

1. Get your token from: https://huggingface.co/settings/tokens
2. Run the following cell to login:


In [ ]:
# Login to Hugging Face (you'll be prompted for your token)
!huggingface-cli login


## Run Complete Pipeline

The complete pipeline includes:
1. **Dataset Generation**: Generate toxic and non-toxic text samples
2. **IRL Training**: Learn reward functions using Bayesian IRL
3. **Spurious Features Analysis**: Analyze learned objectives for spurious patterns

This will use the default Llama-3.2-1B model configuration:


In [ ]:
# Run the complete pipeline
!./run_pipeline.sh complete


## Custom Configuration Examples

### Use Different Models

You can easily change the models used in the pipeline:


In [ ]:
# Example: Use smaller models for faster execution
!./run_pipeline.sh complete \
  toxic_model=EleutherAI/pythia-410m \
  non_toxic_model=ajagota71/pythia-410m-s-nlp-detox-checkpoint-epoch-100


### Run Individual Components

You can also run individual components of the pipeline:


In [ ]:
# Generate datasets only
!./run_pipeline.sh generate


In [ ]:
# Generate datasets with custom parameters
!./run_pipeline.sh generate \
  dataset.train_samples=1000 \
  dataset.test_samples=200 \
  dataset.toxicity_threshold=0.8


In [ ]:
# Train IRL model only (requires existing datasets)
!./run_pipeline.sh train


In [ ]:
# Train IRL model with custom parameters
!./run_pipeline.sh train \
  training.n_steps=3000 \
  training.learning_rate=0.01 \
  training.batch_size=32


In [ ]:
# Run spurious features analysis only (requires trained model)
!./run_pipeline.sh analyze


### RLHF Training

**Important**: RLHF using the IRL reward model is currently only available for the Llama-3.2-1B model. Other models use standard RLHF training.


In [ ]:
# Run RLHF training with default Llama model (includes IRL reward)
!./run_pipeline.sh rlhf


In [ ]:
# Run RLHF with custom parameters
!./run_pipeline.sh rlhf \
  rlhf_config.training.num_train_epochs=2 \
  rlhf_config.training.learning_rate=5e-6


## Results

The pipeline generates several outputs:

### Generated Datasets
- `datasets/*_samples_original.json`: Original model outputs
- `datasets/*_samples_detoxified.json`: Detoxified model outputs
- `datasets/sorted_toxic_dataset_*.json`: Sorted toxic samples
- `datasets/sorted_non_toxic_dataset_*.json`: Sorted non-toxic samples

### Training Results
- `outputs/re_irl/{timestamp}/`: Training outputs directory
- `round_{i}/`: Results for each training round
- `summary.json`: Summary of all rounds
- Various plots and visualizations

### Key Configuration Files
- `configs/full_pipeline.yaml`: Main pipeline configuration
- `configs/dataset.yaml`: Dataset generation settings
- `configs/re_irl_config.yaml`: IRL training parameters
- `configs/rlhf_config.yaml`: RLHF training parameters


## Troubleshooting

### Common Issues

**CUDA Out of Memory:**
```bash
# Use smaller models
!./run_pipeline.sh complete toxic_model=EleutherAI/pythia-70m
```

**Missing Dependencies:**
```bash
!pip install -r requirements.txt
```

**Permission Errors:**
```bash
!chmod +x run_pipeline.sh
```

### Available Models

**Small Models (Fast, Good for Testing):**
- `EleutherAI/pythia-70m`
- `EleutherAI/gpt-neo-125m`
- `HuggingFaceTB/SmolLM-135M`

**Medium Models (Balanced):**
- `EleutherAI/pythia-410m`
- `HuggingFaceTB/SmolLM-360M`

**Large Models (Best Performance):**
- `meta-llama/Llama-3.2-1B` ⭐ **(Recommended for IRL+RLHF pipeline)**
